# wag 🐾 — SFT on Qwen3.5-4B

v2. 8,242 rows against v1's 1,564, and 54% of them are multi-turn rather than 14%, so
this is a different training problem from the one the 2B notebook was written for.

**what changed from v1, and why**

`SEQ_LEN` is 3072, not 1024. `encode` drops anything longer rather than truncating
mid-answer — which is the right call, but at 1024 it quietly drops 171 rows and takes
most of the long-input slice with them. the data tops out at ~2,722 tokens, so 3072
keeps all of it. the collator pads per batch, so the 98% of rows under 620 tokens cost
nothing extra for the headroom.

LoRA is picked by what colab hands you, not set by hand. a 4B full fine-tune needs the
weights, the grads and two fp32 adam moments — call it 48GB before activations — so it
only fits the 80GB A100. on a 40GB card it OOMs partway through the first epoch, which
is a miserable way to find out. `nvidia-smi` decides.

gradient checkpointing stays off. qwen3.5's linear-attention layers raise CheckpointError
on recompute — the recomputed pass sees a different batch shape than the one it saved.
that was true on the 2B and nothing about it changed.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# transformers 5.x for the qwen3_5 arch. upper-bounded because 6.0 will break something,
# lower bound checked against pypi: qwen3_5 lands in 5.15.1, and >=5.16 does not exist yet
# (asked for it once, pip laughed). trl and bitsandbytes are gone — nothing imported them.
!pip install -q -U "transformers>=5.15,<6" "accelerate>=1.10" "datasets>=3.0" "peft>=0.15"


In [ ]:
import json, os, re, sys
from pathlib import Path

import torch
from google.colab import drive

drive.mount("/content/drive")

# ---- knobs -------------------------------------------------------------------
BASE        = "Qwen/Qwen3.5-4B"
EPOCHS      = 3
SEQ_LEN     = 3072           # v2 data maxes out at ~2,722 tokens; 1024 drops 171 rows
BATCH       = 2              # halved from v1: twice the params, three times the length
ACCUM       = 8              # effective batch still 16
SAVE_STEPS  = 100            # 8k rows -> ~1,545 steps/epoch at batch 2 x accum 8
WARMUP_PCT  = 0.05

DRIVE  = Path("/content/drive/MyDrive/wag")
OUTDIR = DRIVE / "ckpt-v2"
DRIVE.mkdir(parents=True, exist_ok=True)

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"gpu: {gpu}  ({vram:.0f} GB)")

# a 4B full fine-tune is ~8GB of bf16 weights + 8GB of grads + 32GB of fp32 adam state
# before a single activation. that fits the 80GB card and nothing else, and colab gives
# out both without asking. so this is measured, not assumed.
USE_LORA = vram < 60
BF16     = "T4" not in gpu
LR       = 2e-4 if USE_LORA else 2e-5
print(f"USE_LORA={USE_LORA}  (full fine-tune wants ~48GB before activations)")
print(f"BF16={BF16}   LR={LR}")
if "T4" in gpu:
    print("!! T4 — no bf16, and a 4B will be painfully slow. prefer A100 or L4.")

## data

upload `data/train.jsonl` (from `gen_bulk.py build`) to Drive, or clone the repo.
the records are already chat-formatted with the system prompt baked in.

In [ ]:
# the repo's own train.jsonl, copied up to Drive from home
TRAIN_JSONL = DRIVE / "train.jsonl"

rows = [json.loads(l) for l in TRAIN_JSONL.open(encoding="utf-8")]
print(f"{len(rows)} training records")

from collections import Counter
print("prompt spread:", dict(Counter(r.get("split_kind", "anchor") for r in rows)))
print("slices       :", dict(Counter(r.get("category", "?") for r in rows).most_common()))
multi = sum(1 for r in rows
            if sum(m["role"] == "assistant" for m in r["messages"]) > 1)
print(f"multi-turn   : {multi} ({100*multi/len(rows):.0f}%)")
print("with <think> :", sum(1 for r in rows if "reasoning_content" in r["messages"][-1]))

# v1 was 1,564 rows and 14% multi-turn. if either of those numbers shows up here, the
# wrong train.jsonl got copied to Drive
assert len(rows) > 5000, f"that looks like v1's data, not v2 ({len(rows)} rows)"

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)

# the qwen3.5 template reads `reasoning_content` off the assistant message and wraps it
# in <think>...</think>. messages without it get an EMPTY think block, which is exactly
# the non-thinking mode we want for the other 85%.
print(tok.apply_chat_template(rows[0]["messages"], tokenize=False))

## masking

train on the assistant turn only. without this the model spends most of its gradient
learning to reproduce the system prompt and the user's question, which is not the job.

the trick: render the conversation twice — once up to the assistant turn, once complete —
and everything before the split point becomes label `-100`.

In [ ]:
IGNORE = -100

def encode(rec):
    msgs = rec["messages"]
    prompt_only = tok.apply_chat_template(
        msgs[:-1], tokenize=False, add_generation_prompt=True
    )
    full = tok.apply_chat_template(msgs, tokenize=False)

    p_ids = tok(prompt_only, add_special_tokens=False)["input_ids"]
    f_ids = tok(full, add_special_tokens=False)["input_ids"]

    if len(f_ids) > SEQ_LEN:
        return None                      # drop rather than truncate mid-answer
    labels = [IGNORE] * len(p_ids) + f_ids[len(p_ids):]
    return {"input_ids": f_ids, "labels": labels[:len(f_ids)],
            "attention_mask": [1] * len(f_ids)}

encoded = [e for e in (encode(r) for r in rows) if e]
print(f"{len(encoded)} usable / {len(rows)}  ({len(rows)-len(encoded)} over {SEQ_LEN} tokens, dropped)")

lens = sorted(len(e["input_ids"]) for e in encoded)
print(f"token length: median {lens[len(lens)//2]}  p95 {lens[int(len(lens)*0.95)]}  max {lens[-1]}")

# sanity: how much of the sequence are we actually training on?
sup = sum(sum(1 for x in e["labels"] if x != IGNORE) for e in encoded)
tot = sum(len(e["labels"]) for e in encoded)
print(f"supervised tokens: {sup}/{tot} ({100*sup/tot:.0f}%)  <- should be roughly 45-70%")

In [ ]:
from dataclasses import dataclass
from datasets import Dataset

ds = Dataset.from_list(encoded).train_test_split(test_size=0.02, seed=20260823)
print(ds)

@dataclass
class PadCollator:
    pad_id: int
    def __call__(self, feats):
        n = max(len(f["input_ids"]) for f in feats)
        out = {"input_ids": [], "labels": [], "attention_mask": []}
        for f in feats:
            gap = n - len(f["input_ids"])
            out["input_ids"].append(f["input_ids"] + [self.pad_id] * gap)
            out["labels"].append(f["labels"] + [IGNORE] * gap)
            out["attention_mask"].append(f["attention_mask"] + [0] * gap)
        return {k: torch.tensor(v) for k, v in out.items()}

collator = PadCollator(tok.pad_token_id or tok.eos_token_id)

## model

Qwen3.5-2B is a **VLM** — it ships vision and video towers. we're training text-only, so
those get frozen: they'd otherwise eat memory and gradient for no reason.

In [ ]:
from transformers import AutoModelForCausalLM

dtype = torch.bfloat16 if BF16 else torch.float16

try:
    from transformers import AutoModelForImageTextToText as _Loader
    model = _Loader.from_pretrained(BASE, dtype=dtype, trust_remote_code=True)
except Exception as e:
    print(f"image-text loader failed ({e}), falling back to causal-lm")
    model = AutoModelForCausalLM.from_pretrained(BASE, dtype=dtype, trust_remote_code=True)

# freeze anything vision-shaped — text-only fine-tune
frozen = 0
for name, p in model.named_parameters():
    if re.search(r"vision|visual|image_|video_|patch_embed", name, re.I):
        p.requires_grad = False
        frozen += p.numel()
print(f"froze {frozen/1e6:.0f}M vision params")

model.config.use_cache = False           # no kv cache while training
# no gradient_checkpointing_enable() here on purpose — TrainingArguments says False and
# qwen3.5's linear-attention layers raise CheckpointError on recompute anyway

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable: {trainable/1e9:.2f}B")

In [ ]:
if USE_LORA:
    from peft import LoraConfig, get_peft_model
    model = get_peft_model(model, LoraConfig(
        r=32, lora_alpha=64, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    ))
    model.print_trainable_parameters()
    LR = 2e-4

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=str(OUTDIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_PCT,   # v5 folded warmup_ratio into this; a float <1 is a ratio
    logging_steps=10,
    save_steps=SAVE_STEPS,
    save_total_limit=3,            # ~12GB a pop once optimizer.pt is counted
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    bf16=BF16,
    fp16=not BF16,
    optim="adamw_torch_fused" if not USE_LORA else "adamw_torch",
    # OFF on purpose. qwen3.5's linear-attention layers don't survive checkpoint
    # recomputation — torch raises CheckpointError because the recomputed pass sees a
    # different batch shape than the one it saved ([4,384] vs [2,209,1]). it also
    # happens to be faster; the 40GB A100 has room without it.
    gradient_checkpointing=False,
    report_to="none",
    seed=20260823,
)

trainer = Trainer(model=model, args=args, train_dataset=ds["train"],
                  eval_dataset=ds["test"], data_collator=collator)

## train

`resume_from_checkpoint=True` picks up the last checkpoint in Drive. if colab kills the
session, re-run every cell from the top and this one continues where it stopped.

In [ ]:
ckpts = sorted(OUTDIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1])) if OUTDIR.exists() else []
resume = bool(ckpts)
print(f"resuming from {ckpts[-1].name}" if resume else "starting fresh")

trainer.train(resume_from_checkpoint=resume)

In [ ]:
# v1 shipped from DRIVE/"wag-final" and those files are still the released model
FINAL = DRIVE / "wag-final-v2"
if USE_LORA:
    model = model.merge_and_unload()      # bake the adapter in so gguf export is simple
model.save_pretrained(FINAL, safe_serialization=True)
tok.save_pretrained(FINAL)
print(f"saved -> {FINAL}")

## smoke test

before you spend time on gguf, check it actually talks like a puppy. the `--no-system`
equivalent below is the important one — an empty system prompt should still give voice,
because 10% of the training set had no system prompt at all.

In [ ]:
model.config.use_cache = True
model.eval()

SYSTEM = ("you are wag, a helpful puppygirl. speak in puppyspeak \u2014 lowercase, soft, "
          "playful. always actually answer the question.")

def ask(prompt, system=SYSTEM, thinking=False, max_new=400):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                   enable_thinking=thinking)
    ids = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new, do_sample=True,
                             temperature=0.7, top_p=0.9,
                             # or it writes the user's next line too — it does emit
                             # <|im_end|>, generate just won't stop on it by default
                             eos_token_id=[i for i in {tok.eos_token_id,
                                                       tok.convert_tokens_to_ids("<|im_end|>")}
                                           if isinstance(i, int) and i >= 0],
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

for p in ["hey wag", "what's the difference between a mutex and a semaphore?", "i got laid off today"]:
    print("=" * 70); print("USER:", p); print("-" * 70); print(ask(p)); print()

print("=" * 70); print("EMPTY SYSTEM PROMPT — voice should survive this")
print("-" * 70); print(ask("hey wag", system=None))

## gguf export

pinned to llama.cpp **v0.2.0** (`5a32f7b`, 2026-08-21) - their first semver-stable tag.
verified at that revision: `conversion/qwen.py:630` registers
`Qwen3_5ForConditionalGeneration` to `Qwen3_5TextModel`, which pulls the text tower out and
drops the vision side, so a text-only fine-tune of the VLM converts without extra surgery.

the mixin there also force-writes `qwen35.rope.dimension_sections` (default `[11, 11, 10, 0]`)
when a checkpoint omits `mrope_section` - the loader treats that field as required, so
without it you get a gguf that builds fine and then refuses to load.

second one of those, found the hard way: the converter sets
`block_count = num_hidden_layers + mtp_num_hidden_layers` = 25, because qwen3.5 normally
ships a multi-token-prediction head. our checkpoint has no mtp head (save_pretrained never
wrote one) but config.json still advertises `mtp_num_hidden_layers: 1`, so you get a header
promising 25 blocks over 24 blocks of tensors and `check_tensor_dims: tensor
'blk.24.attn_norm.weight' not found` at load time. `fix_gguf_blocks.py` re-points those two
u32s at reality; run it on the f16 and the quants inherit it.

note the repo restructured: model classes live in `conversion/` now, `convert_hf_to_gguf.py`
is a thin cli shim over it.

In [ ]:
LLAMA_SHA = "5a32f7b66ef6cfb3e60deea26e3454cc6ad3438c"   # tag v0.2.0

%cd /content
!git clone https://github.com/ggml-org/llama.cpp
%cd llama.cpp
!git checkout {LLAMA_SHA}
!pip install -q -r requirements.txt

# sanity check the pin before burning build time on it
!grep -n 'Qwen3_5ForConditionalGeneration' conversion/qwen.py

!cmake -B build -DGGML_CUDA=OFF > /dev/null && cmake --build build --config Release -j --target llama-quantize > /dev/null
print("built")


In [ ]:
GGUF = DRIVE / "gguf-v2"
GGUF.mkdir(exist_ok=True)

!python convert_hf_to_gguf.py {FINAL} --outfile {GGUF}/wag-f16.gguf --outtype f16

# header says 25 blocks, file has 24 - see the note above. patch before quantizing so the
# quants come out loadable too.
!python {DRIVE}/fix_gguf_blocks.py {GGUF}/wag-f16.gguf

for q in ["q8_0", "q4_k_m"]:
    !./build/bin/llama-quantize {GGUF}/wag-f16.gguf {GGUF}/wag-{q}.gguf {q}

# llama-quantize copies the kv block across, so this should just say "already fine".
# cheap enough to check rather than assume.
!python {DRIVE}/fix_gguf_blocks.py {GGUF}/wag-q8_0.gguf {GGUF}/wag-q4_k_m.gguf

!ls -lh {GGUF}

## eval

scores the 20 held-out prompts for the base model, the shipped 3-epoch model, and whichever
earlier checkpoint you want to argue for. also runs the shipped model with an **empty** system
prompt, which is the real test of whether the voice is in the weights.

heads up: the gguf cell above installs llama.cpp's `requirements.txt`, which **downgrades
transformers to 4.x and numpy to 1.26**. anything after it that touches qwen3.5 will die with
"Transformers does not recognize this architecture" until you put them back — hence the pip
line here. found this the annoying way.

In [ ]:
!pip install -q -U "numpy>=2" "transformers>=5.15,<6"
%cd /content/drive/MyDrive/wag

!python eval.py gen --backend hf --model Qwen/Qwen3.5-2B  --out out_base.jsonl
!python eval.py gen --backend hf --model ./wag-final      --out out_wag3ep.jsonl
!python eval.py gen --backend hf --model ./wag-final --no-system --out out_wag3ep_nosys.jsonl

for f in ["out_base", "out_wag3ep", "out_wag3ep_nosys"]:
    print(f"\n===== {f} =====")
    !python eval.py voice {f}.jsonl | tail -4
